In [2]:
import os
import re
import math
import json
from tqdm import tqdm
import random
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [3]:
MAXIMUM_DATAPOINTS = 10_000

In [4]:
DB = "products-vectorstore"
load_dotenv(override=True)
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [6]:
from pinecone import Pinecone
pc = Pinecone(api_key=pinecone_api_key)

In [7]:
index = pc.Index(DB)

In [8]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['red', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'cyan']

In [ ]:
idx_to_fetch = [f"doc_{i}" for i in range(20000)]
idx_to_fetch = random.sample(idx_to_fetch, MAXIMUM_DATAPOINTS)
print(type(idx_to_fetch),len(idx_to_fetch))

vectors = []
documents = []
categories = []
for i in tqdm(range(0, len(idx_to_fetch), 100)):
    result = index.fetch(ids=idx_to_fetch[i: i+100])
    res = result.vectors
    for j in res.values():
        vectors.append(j['values'])
        documents.append(j['metadata']['description'])
        categories.append(j['metadata']['category'])
vectors = np.array(vectors)
colors = [COLORS[CATEGORIES.index(c)] for c in categories]


<class 'list'> 10000


100%|██████████| 100/100 [00:48<00:00,  2.07it/s]


In [56]:
tsne = TSNE(n_components=2, random_state=42, n_jobs=-1)
reduced_vectors = tsne.fit_transform(vectors)

In [63]:
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    # text=documents,
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(categories, documents)],
    hoverinfo='text',
    marker=dict(size=3, color=colors, opacity=0.7),
)])

fig.update_layout(
    title='2D Pinecone Vectorstore Visualization',
    xaxis_title='x',
    yaxis_title='y',
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()